In [16]:
# Imports
import numpy as np

In [17]:
# Array of r values
dr = 0.5
N = 100
r_values = np.arange(N) * dr + dr

# Defining constants
U = 1
H = 1
k = np.ones(N)
v_parallel = 1

# Bias voltage (equilibrium potential)
phi_0 = np.random.uniform(-1, 1, 100)  # Random values between -1 and 1

# Equilibrium pressure
p_0 = np.random.uniform(-1, 1, 100) 

# Which mode
m = 1

# Matrix Helper Functions

In [20]:
def create_first_derivative_matrix(N, dx):
    matrix = np.zeros((N,N))

    for i in range(1, N-1, 1):
        matrix[i][i+1] = 1 
        matrix[i][i-1] = -1

    matrix[0][0] = -3 
    matrix[0][1] = 4
    matrix[0][2] = -1

    matrix[N-1][N-3] = 1 
    matrix[N-1][N-2] = -4
    matrix[N-1][N-1] = 3

    return matrix / (2 * dx)

def create_second_derivative_matrix(N, dx):
    matrix = np.zeros((N,N))

    for i in range(1, N-1, 1):
        matrix[i][i+1] = 1 
        matrix[i][i] = -2
        matrix[i][i-1] = 1

    matrix[0][0] = 2
    matrix[0][1] = -5
    matrix[0][2] = 4
    matrix[0][3] = -1

    matrix[N-1][N-4] = -1
    matrix[N-1][N-3] = 4
    matrix[N-1][N-2] = -5
    matrix[N-1][N-1] = 2

    return matrix / (dx ** 2)

def create_third_derivative_matrix(N, dx):
    matrix = np.zeros((N,N))

    for i in range(2, N-2, 1):
        matrix[i][i-2] = -1
        matrix[i][i-1] = 2
        matrix[i][i+1] = -2
        matrix[i][i+2] = 1

    matrix = matrix / (2 * (dx ** 3))

    matrix[0][0] = -5
    matrix[0][1] = 18
    matrix[0][2] = -24
    matrix[0][3] = 14
    matrix[0][4] = -3

    matrix[N-1][N-1] = 5
    matrix[N-1][N-2] = -18
    matrix[N-1][N-3] = 24
    matrix[N-1][N-4] = -14
    matrix[N-1][N-5] = 3

    matrix[1][0] = -3
    matrix[1][1] = 10
    matrix[1][2] = -12
    matrix[1][3] = 6
    matrix[1][4] = -1

    matrix[N-2][N-1] = 3
    matrix[N-2][N-2] = -10
    matrix[N-2][N-3] = 12
    matrix[N-2][N-4] = -6
    matrix[N-2][N-5] = 1

    return (matrix / (2 * (dx ** 3)))

def create_m_laplacian(N, dx, m, r_values):
    D1 = create_first_derivative_matrix(N, dx)
    D2 = create_second_derivative_matrix(N, dx)

    return (
        D2
        + np.diag(1 / r_values) @ D1
        - np.diag(m**2 / (r_values**2))
    )

# Dirichlet boundary conditions at edge, regularity boundary conditions at center 
# NOTE: may want to impose boundary conditions for phi[0] later on....
def enforce_bc(N, matrix):
    matrix[N-1][:] = 0
    matrix[N-1][N-1] = 1

    return matrix


## Linear Stability Operator

The eigenvalue problem to be solved is:

$$
\gamma
\begin{pmatrix} \Delta_m & 0 \\ 0 & 1 \end{pmatrix}
\begin{pmatrix} \hat\varphi \\ \hat\rho \end{pmatrix}
=
\begin{pmatrix} \mathcal{L}_{11} & \mathcal{L}_{12} \\ \mathcal{L}_{21} & \mathcal{L}_{22} \end{pmatrix}
\begin{pmatrix} \hat\varphi \\ \hat\rho \end{pmatrix}.
$$

### Linear operators (expanded form)

$$
\mathcal{L}_{11}
  = -im\Omega_0 \Delta_m
    + \frac{im}{r}(\Delta\varphi_0)'
    - \frac{imU}{r}\left(P_0'' + P_0'\,\frac{\partial}{\partial r}\right)
    + \frac{im^3 U}{r^3}\,P_0'
    + H
    - \nu_\parallel \Delta_m,
$$

$$
\mathcal{L}_{12}
  = \frac{imU}{r}\left(\varphi_0''' + \varphi_0''\,\frac{\partial}{\partial r}\right) - 2im\mathcal{K},
$$

$$
\mathcal{L}_{21} = \frac{im}{r}P_0'(r),
$$

$$
\mathcal{L}_{22} = -\nu_\parallel^P - im\Omega_0(r).
$$

where
$$\Omega_0(r) \equiv \frac{1}{r}\frac{\partial \varphi_0}{\partial r}, \quad \text{where } \varphi_0 \text{ is the equilibrium potential.}$$

and 

$$
(\Delta\varphi_0)' = \varphi_0''' + \frac{\varphi_0''}{r} - \frac{\varphi_0'}{r^2}.
$$

In [23]:
# Initalizing matrices
D1 = create_first_derivative_matrix(N, dr)
D2 = create_second_derivative_matrix(N, dr)
D3 = create_third_derivative_matrix(N, dr)
Dm = create_m_laplacian(N, dr, m, r_values)

diag_r = np.diag(r_values)
diag_div_r = np.diag(1 / r_values)

omega_0 = diag_div_r @ np.diag(D1 @ phi_0)
D_phi_prime = (D3 + diag_div_r @ D2 - np.diag(1 / (r_values ** 2)) @ D1) @ phi_0

# Defining opereators
L11 = -1j * m * np.diag(omega_0 ) @ Dm + 1j * m * diag_div_r @ np.diag(D_phi_prime) - 1j * m * U * diag_div_r @ (np.diag(D2 @ p_0) + np.diag(D1 @ p_0) @ D1) + 1j * (m ** 3) * U * np.diag(1 / (r_values ** 3)) @ np.diag(D1 @ p_0) + np.diag(np.ones(N) * H) - v_parallel * Dm
L12 = 1j * m * U * diag_div_r @ (np.diag(D3 @ phi_0) + np.diag(D2 @ phi_0) @ D1) - 2j * m * np.diag(k)
L21 = 1j * m * np.diag(1 / r_values) @ np.diag(D1 @ p_0)
L22 = -np.diag(np.ones(N) * v_parallel) - (1j * m * np.diag(omega_0))

In [24]:
# Reduce eigenvalue problem to gamma * Ax = Bx
A = np.block([
    [L11, L12],
    [L21, L22]
])

B = np.block([
    [Dm, np.zeros((N,N))],
    [np.zeros((N,N)), np.ones((N,N))]
])

In [27]:
from scipy.linalg import eig
import numpy as np

eigenvalues, eigenvectors = eig(B, A)

# Each column of eigenvectors is an eigenvector
x0 = eigenvectors[:, 0]  # eigenvector for eigenvalues[0]

In [29]:
eigenvalues.shape

(200,)